# Parameter-Efficient Adaptation of Geospatial Foundation Models with Multi-Sensor Synergy for Forest Carbon Mapping under Limited Plot Inventory in Karst Terrains
### Full Algorithmic Reproducibility & Interactive Walkthrough Notebook
**Repository:** [https://github.com/lethithuhong1302/gfm-forest-carbon](https://github.com/lethithuhong1302/gfm-forest-carbon)  
**Sensors (41 Channels):** Optical (6 HLS Reconstructed), Sentinel-1 C-band (4), ALOS-2 L-band (2), GLCM Textures (8), Spectral Indices (20), SRTM DEM Topography (1)  
**Proposed Framework:** Pre-trained **NASA-IBM Prithvi-100M ViT** with **LoRA PEFT** ($r=8, \alpha=16$) + **CNN Auxiliary Encoder** + **Gated Cross-Modal Fusion** + **Dense Regression Decoder**  
**Physical Carbon Derivation:** $\text{Carbon} = 0.47 \times \text{AGB}$ (IPCC Guidelines)


## 1. Environment Verification & Dependency Setup
Verify GPU acceleration (NVIDIA T4, L4, V100, or A100) and install Python dependencies.


In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"[+] Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"[+] VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("[!] GPU not detected. Running on CPU mode.")


## 2. Dataset Audit & Multi-Sensor Visualization
Audit the bundled anonymized toy dataset (128x128 patches) and render multi-sensor previews.


In [ ]:
!python scripts/00_audit_dataset.py
!python scripts/01_visualize_patches.py

from IPython.display import Image
Image("experiments_output/figures/multimodal_patch_preview.png")


## 3. GFM Architectural Compatibility & PEFT Parameter Analysis
Check token sequence dimensions (64 patches, 768 dim) and contrast parameter efficiency between:
- Full Fine-Tuning (86.2M params, 100%)
- Last-2 Blocks Fine-Tuning (14.2M params, 16.4%)
- LoRA Low-Rank Adaptation (295K params, **0.34%**)
- Proposed Frozen Backbone + Multimodal Adapter (4.1M params, **4.78%**)


In [ ]:
!python scripts/04_verify_compatibility.py


## 4. Prithvi Optical Fine-Tuning Pipeline
Fine-tune the optical foundation model using Parameter-Efficient LoRA ($r=8, \alpha=16.0$) with Epoch-0 protection and validation nRMSE checkpoint selection.


In [ ]:
!python scripts/05_train_optical_lora.py --epochs 15 --peft-mode lora --lora-r 8 --batch-size 4


## 5. Two-Stage Multimodal Adapter Training (41 Channels with DEM)
Warm-start from the optical checkpoint, freeze backbone weights, and train the cross-modal auxiliary adapter.


In [ ]:
!python scripts/06_train_multimodal_adapter.py --scenario S7_Full_Fusion --epochs 15 --fusion-only-epochs 3 --batch-size 4


## 6. Sensor Modality Ablation Benchmark
Benchmark sensor contributions across Scenarios S1 (Optical) to S7 (Full Multimodal with Topography).


In [ ]:
!python scripts/07_ablation_study.py --epochs 5 --batch-size 4


## 7. Explainable AI & Uncertainty Quantification
Calculate group permutation feature importance and evaluate residual error distributions.


In [ ]:
!python scripts/09_xai_analysis.py --scenario S7_Full_Fusion

Image("experiments_output/figures/permutation_importance.png")


## 8. Export IEEE Publication Tables
Export CSV and LaTeX tables formatted for IEEE Transactions submission.


In [ ]:
!python scripts/10_export_results.py

import pandas as pd
df_table1 = pd.read_csv("experiments_output/tables/table1_main_model_comparison.csv")
display(df_table1)
